In [1]:
import pandas as pd
import os
from pathlib import Path
from glob import glob
from langchain.chat_models import init_chat_model
from time import time
from dotenv import load_dotenv
from tqdm import tqdm
load_dotenv()

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Get all .java and .py files from the MS repo clusters directory
BASE_DIR = Path("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/ms_repo_callgraph_clusters")
java_files = glob(str(BASE_DIR / "**/*.java"), recursive=True)
py_files = glob(str(BASE_DIR / "**/*.py"), recursive=True)
all_files = java_files + py_files

print(f"Found {len(java_files)} .java files and {len(py_files)} .py files")
print(f"Total files to process: {len(all_files)}")

def load_code(path):
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as file:
            return file.read()
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return ""

# Create DataFrame with file paths and code
data = []
for file_path in all_files:
    code = load_code(file_path)
    if code.strip():  # Only include files with content
        data.append({
            'file': file_path,
            'code': code
        })

df_files = pd.DataFrame(data)
print(f"Loaded {len(df_files)} files with content")

Found 954 .java files and 156 .py files
Total files to process: 1110
Loaded 1110 files with content


In [3]:
# Initialize LLM (using Gemini as it's more cost-effective for this task)
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

In [4]:
# Prompt for generating descriptions of individual code files
prompt = """\
You are an expert code analyst. Generate a concise and informative description of this code file in 1-2 sentences. 
Focus on what the code does, its main functionality, and any notable patterns or approaches used.
Be specific about the code's purpose and avoid generic statements.

Code file:
{code}
"""

In [5]:
def generate_file_description(code: str, llm, max_retries=3) -> str:
    for attempt in range(max_retries):
        try:
            response = llm.invoke(prompt.format(code=code))
            description = response.content.strip()
            if description:
                return description
        except Exception as e:
            if attempt < max_retries - 1:
                import time
                time.sleep(1)
            else:
                print(f"Error generating description after {max_retries} attempts: {e}")
    
    return "Failed to generate description"

In [6]:
# Output file path
output_file = "result/ms_community_descriptions.csv"

# Load existing descriptions if file exists
if os.path.exists(output_file):
    descriptions_df = pd.read_csv(output_file)
    processed_files = set(descriptions_df['file'].values)
    print(f"Found existing descriptions for {len(processed_files)} files")
else:
    descriptions_df = pd.DataFrame(columns=['file', 'code', 'description'])
    processed_files = set()

# Process each file
# Process files with progress bar
for index, row in tqdm(df_files.iterrows(), total=len(df_files), desc="Processing files"):
    file_path = row['file']
    code = row['code']
    
    # Skip if already processed
    if file_path in processed_files:
        continue
    
    # Generate description
    description = generate_file_description(code, llm)
    
    # Add to DataFrame
    new_row = {
        'file': file_path,
        'code': code,
        'description': description
    }
    descriptions_df = pd.concat([descriptions_df, pd.DataFrame([new_row])], ignore_index=True)
    
    # Save periodically (every 10 files)
    if len(descriptions_df) % 10 == 0:
        descriptions_df.to_csv(output_file, index=False)

# Final save
descriptions_df.to_csv(output_file, index=False)
print(f"\nCompleted! Processed {len(descriptions_df)} files. Saved to {output_file}")

Found existing descriptions for 910 files


Processing files: 100%|██████████| 1110/1110 [16:05<00:00,  1.15it/s]



Completed! Processed 1110 files. Saved to result/ms_community_descriptions.csv


In [9]:
descriptions_df['file'] = descriptions_df['file'].apply(lambda x: x.replace('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/', 'https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project'))

In [10]:

descriptions_df.to_csv(output_file, index=False)

In [12]:
descriptions_df.iloc[0]['file']

'https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Projectresult/ms_repo_callgraph_clusters/saigonparking_java/cluster_11.java'

In [11]:
# Display sample of results
descriptions_df.head()

,file,code,description
0,https://github.com/HasinthakaPiyumal/AI-Patter...,// Cluster 11\n\npackage com.bht.saigonparking...,This file defines a suite of custom Java annot...
1,https://github.com/HasinthakaPiyumal/AI-Patter...,// Cluster 19\n\npackage com.bht.saigonparking...,This abstract Spring configuration class estab...
2,https://github.com/HasinthakaPiyumal/AI-Patter...,// Cluster 31\n\npackage com.bht.saigonparking...,This Spring component defines HTML email templ...
3,https://github.com/HasinthakaPiyumal/AI-Patter...,// Cluster 7\n\npackage com.bht.saigonparking....,This Spring component configures RabbitMQ for ...
4,https://github.com/HasinthakaPiyumal/AI-Patter...,// Cluster 9\n\npackage com.bht.saigonparking....,This code defines two Spring MVC controllers f...
